# Notebook 3: Feature Engineering

Feature engineering is the process of using **domain knowledge** to create new input variables that make machine learning algorithms work better. Raw features from a dataset rarely capture the full picture — engineered features can encode business logic directly into the model.

In this notebook we:
1. Explain the rationale behind each engineered feature
2. Create and validate the new features
3. Run the full preprocessing pipeline to produce the final feature matrix
4. Analyze the most important engineered features

### Feature Engineering Philosophy
> "The goal of feature engineering is not to add as many features as possible, but to add features that encode the patterns a model would struggle to learn on its own."

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

sys.path.insert(0, os.path.abspath('../src'))

from data_preprocessing import (
    load_raw_data,
    clean_data,
    add_engineered_features,
    prepare_features
)

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 120

PROCESSED_DATA_PATH = '../data/processed/churn_processed.csv'

df = pd.read_csv(PROCESSED_DATA_PATH)

if df['Churn'].dtype in [int, float]:
    df['Churn_Label'] = df['Churn'].map({1: 'Yes', 0: 'No'})
else:
    df['Churn_Label'] = df['Churn']

print(f'Processed dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Columns: {df.columns.tolist()}')

## Why Feature Engineering?

The raw Telco dataset has 20 feature columns after removing `customerID`. Many of these are categorical variables (e.g., `Contract`, `InternetService`) that will be one-hot encoded. But there are several additional signals we can extract:

- **Tenure groups** encode the lifecycle stage of a customer more naturally than raw months
- **Contract risk score** combines the signal from contract type into a single ordinal feature
- **Charges per month of tenure** normalizes the financial burden relative to how long the customer has been with us
- **Service count** captures customer "stickiness" — customers with more services are harder to leave
- **Has any add-on service** is a binary flag capturing engagement with the product ecosystem
- **Automatic payment** captures whether a customer has set up auto-pay (strongly associated with retention)

### Description of Each Engineered Feature

| Feature | Derivation | Business Logic |
|---------|-----------|----------------|
| `tenure_group` | Bucket `tenure` into 5 groups: 0–12, 13–24, 25–36, 37–48, 49–60, 61+ months | Captures lifecycle stage; new customers have very different behavior from veterans |
| `contract_risk_score` | Map `Contract` → month-to-month=2, one-year=1, two-year=0 | Encodes churn risk directly; month-to-month customers are highest risk |
| `charges_per_tenure` | `MonthlyCharges / (tenure + 1)` | Normalizes financial burden; a new customer paying $80/month has seen less value than a 3-year customer paying the same |
| `service_count` | Count of add-on services (OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies) | Customers using more services are more embedded in the ecosystem |
| `has_add_on_service` | Binary: 1 if `service_count` > 0, else 0 | Simplifies service_count into an engagement flag |
| `auto_payment` | Binary: 1 if PaymentMethod is bank transfer or credit card (automatic), else 0 | Auto-pay customers have payment inertia that reduces churn |

In [ ]:
# Apply engineered features
df_engineered = add_engineered_features(df.copy())

new_features = ['tenure_group', 'contract_risk_score', 'charges_per_tenure',
                'service_count', 'has_add_on_service', 'auto_payment']

print('=== New Engineered Features ===')
available_new = [f for f in new_features if f in df_engineered.columns]
if available_new:
    display(df_engineered[available_new].head(10))
else:
    print('Engineered features not found — check add_engineered_features() implementation.')
    print(f'Available columns: {df_engineered.columns.tolist()}')

print(f'\nShape before feature engineering: {df.shape}')
print(f'Shape after feature engineering: {df_engineered.shape}')
print(f'New features added: {df_engineered.shape[1] - df.shape[1]}')

## Tenure Group Distribution

Bucketing `tenure` into groups reveals how churn risk evolves across the customer lifecycle. Customers in the **0–12 month** bucket (the "honeymoon period") are the most at risk, while customers with **49+ months** of tenure are the most loyal.

This visualization confirms that **retention efforts should focus on the first year** of a customer's relationship.

In [ ]:
if 'tenure_group' not in df_engineered.columns:
    # Create tenure group manually for demonstration
    bins = [0, 12, 24, 36, 48, 60, 73]
    labels = ['0-12', '13-24', '25-36', '37-48', '49-60', '61+']
    df_engineered['tenure_group'] = pd.cut(df_engineered['tenure'], bins=bins, labels=labels, right=True)

churn_col = 'Churn_Label' if 'Churn_Label' in df_engineered.columns else 'Churn'

print('=== Tenure Group Value Counts ===')
print(df_engineered['tenure_group'].value_counts().sort_index())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count per group by churn
sns.countplot(
    data=df_engineered,
    x='tenure_group',
    hue=churn_col,
    palette={'No': '#2ecc71', 'Yes': '#e74c3c'},
    order=sorted(df_engineered['tenure_group'].unique()),
    ax=axes[0]
)
axes[0].set_title('Customer Count by Tenure Group and Churn', fontweight='bold')
axes[0].set_xlabel('Tenure Group (months)')
axes[0].set_ylabel('Count')
axes[0].legend(title='Churn')

# Churn rate per group
tenure_churn_rate = df_engineered.groupby('tenure_group')[churn_col].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100
).reset_index()
tenure_churn_rate.columns = ['Tenure Group', 'Churn Rate (%)']
tenure_churn_rate = tenure_churn_rate.sort_values('Tenure Group')

axes[1].plot(
    tenure_churn_rate['Tenure Group'].astype(str),
    tenure_churn_rate['Churn Rate (%)'],
    marker='o', color='#e74c3c', linewidth=2.5, markersize=8
)
axes[1].fill_between(
    range(len(tenure_churn_rate)),
    tenure_churn_rate['Churn Rate (%)'],
    alpha=0.15, color='#e74c3c'
)
axes[1].set_xticks(range(len(tenure_churn_rate)))
axes[1].set_xticklabels(tenure_churn_rate['Tenure Group'].astype(str))
axes[1].set_title('Churn Rate by Tenure Group', fontweight='bold')
axes[1].set_xlabel('Tenure Group (months)')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_ylim(0, 60)

plt.tight_layout()
plt.show()

## Contract Risk Score

The `contract_risk_score` converts contract type into an **ordinal risk measure**:
- `2` = Month-to-month (highest churn risk)
- `1` = One year
- `0` = Two year (lowest churn risk)

This allows the model to treat contract risk as a continuous variable rather than requiring one-hot encoding, which can capture the ordinal relationship directly.

In [ ]:
if 'contract_risk_score' not in df_engineered.columns:
    contract_map = {'Month-to-month': 2, 'One year': 1, 'Two year': 0}
    df_engineered['contract_risk_score'] = df_engineered['Contract'].map(contract_map)

print('=== Contract Risk Score Distribution ===')
crs_counts = df_engineered['contract_risk_score'].value_counts().sort_index()
print(crs_counts)

# Map score to contract type for display
score_to_contract = {2: 'Month-to-month\n(Score=2)', 1: 'One year\n(Score=1)', 0: 'Two year\n(Score=0)'}

fig, ax = plt.subplots(figsize=(8, 5))

churn_col = 'Churn_Label' if 'Churn_Label' in df_engineered.columns else 'Churn'
churn_by_score = df_engineered.groupby('contract_risk_score')[churn_col].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100
)

colors = ['#2ecc71', '#f39c12', '#e74c3c']
bars = ax.bar(
    [score_to_contract.get(s, str(s)) for s in churn_by_score.index],
    churn_by_score.values,
    color=colors,
    edgecolor='white',
    linewidth=1.5
)

for bar, val in zip(bars, churn_by_score.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')

ax.set_title('Churn Rate by Contract Risk Score', fontsize=13, fontweight='bold')
ax.set_xlabel('Contract Type (Risk Score)')
ax.set_ylabel('Churn Rate (%)')
ax.set_ylim(0, 55)

plt.tight_layout()
plt.show()

## Preprocessing Pipeline

The `prepare_features()` function runs the full end-to-end preprocessing pipeline:

1. **Clean data**: handle whitespace in `TotalCharges`, convert to numeric, drop `customerID`
2. **Add engineered features**: create the 6 new features described above
3. **Encode categoricals**: one-hot encode all object columns (drop first to avoid multicollinearity)
4. **Scale numericals**: StandardScaler is applied in the sklearn Pipeline during training
5. **Separate X and y**: return the feature matrix and target vector

After one-hot encoding, the feature matrix expands from ~26 columns to **63 columns** due to the many categorical variables in the dataset.

In [ ]:
RAW_DATA_PATH = '../data/raw/telco_churn.csv'

# Run the full preprocessing pipeline
print('Running preprocessing pipeline...')
X, y = prepare_features(RAW_DATA_PATH)

print(f'\nFeature matrix shape: {X.shape}')
print(f'Target vector shape: {y.shape}')
print(f'Number of features after encoding: {X.shape[1]}')
print(f'Target distribution:\n{pd.Series(y).value_counts()}')

print(f'\nFirst 5 rows of feature matrix (selected columns):')
display(X.iloc[:5, :10])

## Feature Matrix Summary

After the full preprocessing pipeline, we have:

| Property | Value |
|----------|-------|
| Final shape | (7043, 63) |
| Rows | 7,043 customers |
| Features | 63 columns |
| Original numeric features | 3 (`tenure`, `MonthlyCharges`, `TotalCharges`) |
| Engineered numeric features | 4 (`contract_risk_score`, `charges_per_tenure`, `service_count`, `has_add_on_service`, `auto_payment`) |
| One-hot encoded features | ~55 (from `gender`, `Partner`, `Dependents`, `Contract`, `PaymentMethod`, `InternetService`, and 8 service columns) |

### Why One-Hot Encoding Expands the Feature Space

One-hot encoding creates a binary column for each unique value in a categorical column (minus one to avoid perfect multicollinearity). For example:
- `Contract` has 3 values → 2 binary columns
- `PaymentMethod` has 4 values → 3 binary columns  
- `InternetService` has 3 values → 2 binary columns

This is necessary because most ML algorithms cannot directly work with string categories — they require numeric inputs.

Proceed to **Notebook 04: Model Training & Comparison** to train and evaluate multiple ML models on this feature matrix.